In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import re
from pathlib import Path

## Explanation

For this file, I am looking for the real performance scores for each player after inital four years. As we discussed, the player who has less than 5 years will be dropped. In the next cell:
All of this results only used data file: Career_totals_targets_with_performance_analysis in data folder.

1. First, I build an scores based on the PCA features (usig the feature for each player after 4 years, then taking the average for the rest for each feature. And then we will have one player as row with averaged_feature_1, averaged_feature_2, ... for this player) Do this for all players

2. Put in the PCA, and let PCA to find the weights of each ave feature. (weights can be founding in loading fucntion)

3. Find PCA_PC1 scores, formula shold be PC1_scores = PC1_weights*ave_feature_1+.......

4. Direct call performance_analysis file, the last three column is also what we need. Do the same things, taking the average of those variables after 4 years for each player.

5. Save the PC1_scores with additional three scores together in player_scores_with_realized_score_z

In [4]:
# =========================================================
# 1. Read the career totals file
# =========================================================
PROJECT_ROOT = Path().resolve().parents[0]
DATA_DIR = PROJECT_ROOT / "data"
df = pd.read_csv(DATA_DIR / "career_totals_targets_with_performance_analysis.csv")

# career_path = "career_totals_targets_with_performance_analysis.csv"
# df = pd.read_csv(career_path)

print("Original shape:", df.shape)
print(df.head())


# =========================================================
# 2. Basic cleaning
# =========================================================
keep_cols = [
    "PLAYER_ID", "PLAYER_NAME", "SEASON_ID", "TEAM_ABBREVIATION",
    "PLAYER_AGE", "GP", "GS", "MIN",
    "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

df = df[keep_cols].copy()

numeric_cols = [
    "PLAYER_AGE", "GP", "GS", "MIN",
    "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["PLAYER_ID", "PLAYER_NAME", "SEASON_ID"]).copy()

print("\nAfter basic cleaning:", df.shape)


# =========================================================
# 3. Resolve duplicate player-season rows
#    Rule:
#    - If TOT row exists, keep TOT
#    - Otherwise aggregate across team rows
# =========================================================
group_cols = ["PLAYER_ID", "PLAYER_NAME", "SEASON_ID"]

career_season_rows = []

for _, group in df.groupby(group_cols):
    if (group["TEAM_ABBREVIATION"] == "TOT").any():
        row = group.loc[group["TEAM_ABBREVIATION"] == "TOT"].iloc[0].copy()
        career_season_rows.append(row.to_dict())
    else:
        out = {}
        out["PLAYER_ID"] = group["PLAYER_ID"].iloc[0]
        out["PLAYER_NAME"] = group["PLAYER_NAME"].iloc[0]
        out["SEASON_ID"] = group["SEASON_ID"].iloc[0]
        out["TEAM_ABBREVIATION"] = "AGG"
        out["PLAYER_AGE"] = group["PLAYER_AGE"].max()

        sum_cols = [
            "GP", "GS", "MIN", "FGM", "FGA", "FG3M", "FG3A",
            "FTM", "FTA", "OREB", "DREB", "REB", "AST", "STL",
            "BLK", "TOV", "PF", "PTS"
        ]

        for col in sum_cols:
            out[col] = group[col].sum(skipna=True)

        out["FG_PCT"] = out["FGM"] / out["FGA"] if out["FGA"] > 0 else np.nan
        out["FG3_PCT"] = out["FG3M"] / out["FG3A"] if out["FG3A"] > 0 else np.nan
        out["FT_PCT"] = out["FTM"] / out["FTA"] if out["FTA"] > 0 else np.nan

        career_season_rows.append(out)

career_season = pd.DataFrame(career_season_rows)

print("\nAfter collapsing player-season duplicates:", career_season.shape)


# =========================================================
# 4. Create career year
# =========================================================
career_season["season_start_year"] = career_season["SEASON_ID"].astype(str).str[:4].astype(int)

career_season = career_season.sort_values(
    ["PLAYER_ID", "season_start_year"]
).reset_index(drop=True)

career_season["career_year"] = (
    career_season.groupby("PLAYER_ID").cumcount() + 1
)

print("\nCareer-year check:")
print(career_season[["PLAYER_ID", "PLAYER_NAME", "SEASON_ID", "career_year"]].head(20))


# =========================================================
# 5. Keep players with career length >= 5
#    Then keep all seasons from career year 5 onward
# =========================================================
career_lengths = (
    career_season.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    .size()
    .rename(columns={"size": "career_length"})
)

career_season = career_season.merge(
    career_lengths,
    on=["PLAYER_ID", "PLAYER_NAME"],
    how="left"
)

# Keep players who played at least 5 seasons
later = career_season[
    (career_season["career_length"] >= 5) &
    (career_season["career_year"] >= 5)
].copy()

print("\nLater-career rows kept:", later.shape)
print("Unique players kept:", later["PLAYER_ID"].nunique())

# =========================================================
# 6. Build player-level summary over all later-career seasons
#    (career year 5 to final year)
# =========================================================
summary_cols = [
    "GP", "GS", "MIN",
    "FGM", "FGA", "FG_PCT",
    "FG3M", "FG3A", "FG3_PCT",
    "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB",
    "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

player_5plus = (
    later.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)[summary_cols]
    .mean()
)

n_later_years = (
    later.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    .size()
    .rename(columns={"size": "n_years_5plus"})
)

player_5plus = player_5plus.merge(
    n_later_years,
    on=["PLAYER_ID", "PLAYER_NAME"],
    how="left"
)

rename_map = {c: f"{c}_5plus_avg" for c in summary_cols}
player_5plus = player_5plus.rename(columns=rename_map)


# =========================================================
# 7. Choose PCA features
# =========================================================
pca_features = [
    "GP_5plus_avg",
    "GS_5plus_avg",
    "MIN_5plus_avg",
    "PTS_5plus_avg",
    "REB_5plus_avg",
    "AST_5plus_avg",
    "STL_5plus_avg",
    "BLK_5plus_avg",
    "FG_PCT_5plus_avg",
    "FG3_PCT_5plus_avg",
    "FT_PCT_5plus_avg"
]

pca_df = player_5plus[["PLAYER_ID", "PLAYER_NAME", "n_years_5plus"] + pca_features].copy()

# ---------------------------------------------------------
# IMPORTANT FIX:
# Do NOT drop players because of missing PCA features.
# Fill missing values with column medians instead.
# ---------------------------------------------------------
missing_before = pca_df[pca_features].isna().sum()
# print("\nMissing values before filling:")
# print(missing_before)

for col in pca_features:
    pca_df[col] = pca_df[col].fillna(pca_df[col].median())

missing_after = pca_df[pca_features].isna().sum()


# =========================================================
# 8. Standardize PCA features
# =========================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df[pca_features])

X_scaled_df = pd.DataFrame(X_scaled, columns=pca_features)

# print("\nScaled feature means (should be near 0):")
# print(X_scaled_df.mean().round(4))

# print("\nScaled feature stds (should be near 1):")
# print(X_scaled_df.std(ddof=0).round(4))


# =========================================================
# 9. Run PCA
# =========================================================
pca = PCA(n_components=len(pca_features), random_state=42)
X_pca = pca.fit_transform(X_scaled)

for i in range(X_pca.shape[1]):
    pca_df[f"PC{i+1}"] = X_pca[:, i]

explained_var = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(pca.explained_variance_ratio_))],
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_explained_variance": np.cumsum(pca.explained_variance_ratio_)
})

# print("\nExplained variance:")
# print(explained_var)


# =========================================================
# 10. Inspect loadings
# =========================================================
loadings = pd.DataFrame(
    pca.components_.T,
    index=pca_features,
    columns=[f"PC{i+1}" for i in range(len(pca_features))]
)

print("\nPC1 loadings:")
print(loadings["PC1"].sort_values(ascending=False))


# =========================================================
# 11. Define realized_score from PC1
# =========================================================
pc1 = pca_df["PC1"].copy()

corr_with_min = np.corrcoef(pc1, pca_df["MIN_5plus_avg"])[0, 1]
corr_with_pts = np.corrcoef(pc1, pca_df["PTS_5plus_avg"])[0, 1]

print("\nCorrelation of PC1 with MIN_5plus_avg:", round(corr_with_min, 4))
print("Correlation of PC1 with PTS_5plus_avg:", round(corr_with_pts, 4))

if (corr_with_min < 0) and (corr_with_pts < 0):
    pca_df["realized_score"] = -pca_df["PC1"]
    sign_flipped = True
else:
    pca_df["realized_score"] = pca_df["PC1"]
    sign_flipped = False

# print("\nSign flipped?", sign_flipped)


# =========================================================
# 12. Standardize realized_score
# =========================================================
pca_df["realized_score_z"] = (
    (pca_df["realized_score"] - pca_df["realized_score"].mean()) /
    pca_df["realized_score"].std(ddof=0)
)

# print("\nRealized score summary:")
# print(pca_df["realized_score_z"].describe())


# =========================================================
# 13. Merge realized score back
# =========================================================
realized_scores = pca_df[[
    "PLAYER_ID", "PLAYER_NAME", "n_years_5plus",
    "realized_score", "realized_score_z"
]].copy()

final_realized = player_5plus.merge(
    realized_scores,
    on=["PLAYER_ID", "PLAYER_NAME", "n_years_5plus"],
    how="inner"
)

Original shape: (9384, 31)
   PLAYER_ID SEASON_ID  LEAGUE_ID     TEAM_ID TEAM_ABBREVIATION  PLAYER_AGE  \
0    1629627   2019-20          0  1610612740               NOP        19.0   
1    1629627   2020-21          0  1610612740               NOP        20.0   
2    1629627   2022-23          0  1610612740               NOP        22.0   
3    1629627   2023-24          0  1610612740               NOP        23.0   
4    1629627   2024-25          0  1610612740               NOP        24.0   

   GP  GS   MIN  FGM  ...  AST  STL  BLK  TOV   PF   PTS      PLAYER_NAME  \
0  24  24   668  210  ...   50   16    9   59   42   540  Zion Williamson   
1  61  61  2026  634  ...  226   57   39  167  135  1647  Zion Williamson   
2  29  29   956  285  ...  133   32   16   99   65   754  Zion Williamson   
3  70  70  2207  624  ...  352   77   47  193  159  1601  Zion Williamson   
4  30  30   857  288  ...  159   37   27   90   82   737  Zion Williamson   

     TS_PCT       VPS  GmSc_Per48  

In [5]:
# 1. Read data
df = pd.read_csv(DATA_DIR / "career_totals_targets_with_performance_analysis.csv")

# 2. Keep only needed columns
df_use = df[[
    "PLAYER_ID",
    "PLAYER_NAME",
    "SEASON_ID",
    "TEAM_ABBREVIATION",
    "TS_PCT",
    "VPS",
    "GmSc_Per48"
]].copy()

# 3. Make sure TOT row is kept when a player has multiple team rows in the same season
#    Rule:
#    - if TOT exists for that player-season, keep TOT
#    - otherwise keep the first row for that player-season
df_use["is_tot"] = (df_use["TEAM_ABBREVIATION"] == "TOT").astype(int)

df_use = (
    df_use
    .sort_values(["PLAYER_ID", "SEASON_ID", "is_tot"], ascending=[True, True, False])
    .drop_duplicates(subset=["PLAYER_ID", "SEASON_ID"], keep="first")
    .drop(columns="is_tot")
    .reset_index(drop=True)
)

# 4. Sort by player and season
df_use = df_use.sort_values(["PLAYER_ID", "SEASON_ID"]).reset_index(drop=True)

# 5. Create career year based on UNIQUE seasons
df_use["career_year"] = df_use.groupby("PLAYER_ID").cumcount() + 1

# 6. Count unique seasons for each player
season_count = (
    df_use.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    .size()
    .rename(columns={"size": "n_unique_seasons"})
)

# 7. Keep only players with more than 4 unique seasons
players_gt4 = season_count.loc[season_count["n_unique_seasons"] > 4, "PLAYER_ID"]

df_use = df_use[df_use["PLAYER_ID"].isin(players_gt4)].copy()

# 8. Keep only later seasons: career year >= 5
df_later = df_use[df_use["career_year"] >= 5].copy()

# 9. Average the target variables over later seasons
player_scores = (
    df_later
    .groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    .agg(
        TS_PCT_score=("TS_PCT", "mean"),
        VPS_score=("VPS", "mean"),
        GmSc_Per48_score=("GmSc_Per48", "mean"),
        later_years_used=("career_year", "count")
    )
)

# 10. Optional: sort result
player_scores = player_scores.sort_values("PLAYER_NAME").reset_index(drop=True)

In [ ]:
# Keep only the columns you need
df_final = final_realized[["PLAYER_ID", "realized_score_z"]].copy()

# Merge into player_scores
player_scores = player_scores.merge(
    df_final,
    on="PLAYER_ID",
    how="left"
)

# Save
output_path = DATA_DIR / "player_scores_with_realized_score_z.csv"
player_scores.to_csv(output_path, index=False)

print(f"Saved combined dataset to: {output_path}")
print(f"Final shape: {player_scores.shape}")


Saved combined dataset to: C:\Users\27946\OneDrive\桌面\Git Clone\STAT-946-Case-Study-3\data\player_scores_with_realized_score_z.csv
Final shape: (655, 7)


In [8]:
print(player_scores)

     PLAYER_ID      PLAYER_NAME  TS_PCT_score  VPS_score  GmSc_Per48_score  \
0       201985         AJ Price      0.459974   1.081828         11.933333   
1       201166     Aaron Brooks      0.519166   1.042882         12.460425   
2       203932     Aaron Gordon      0.587510   1.585852         19.815130   
3       201189       Aaron Gray      0.511780   1.025159          8.601351   
4      1628988    Aaron Holiday      0.561510   1.170613         13.219019   
..         ...              ...           ...        ...               ...   
650    1628380     Zach Collins      0.608635   1.451200         20.173683   
651     203897      Zach LaVine      0.602410   1.351205         22.515632   
652       2216    Zach Randolph      0.519454   1.287924         18.936302   
653       2585    Zaza Pachulia      0.547838   1.308455         14.251055   
654    1629627  Zion Williamson      0.618359   1.653466         31.227671   

     later_years_used  realized_score_z  
0                   2